In [1]:
from google import genai
from google.genai import types
import pandas as pd
import google.generativeai as genai
import json
import os.path

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def main():
    today = datetime.today()
    first_day_this_month = today.replace(day=1)

    # Subtract one day to get the last day of the previous month
    last_day_prev_month = first_day_this_month - timedelta(days=1)
    
    df = extract_data_with_gemini(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\PROFILE CAPEX RKAP 1565 (5.  Mar 2026)c-2-4 (CHECK).xlsx")
    print(df)
    
    if df is not None and not df.empty:
        print("Data extracted successfully.")
        
        creds = connect_to_sheet()
        
        # Append Raw Data
        update_sheet('1ckgGhsbbHGYfpqXtak_EG79ElY00bTvAADbNySRMa1s', 'Detail Capex', None, creds, df)
        
    else:
        print("Failed to extract data. Check Gemini response or JSON format.")
 
    
def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def get_excel_data(address, sheet_name=0, usecols=None, skiprows=0, nrows=None, header=None):
    df = pd.read_excel(
        address,
        sheet_name=sheet_name,
        usecols=usecols,
        skiprows=skiprows,
        nrows=nrows,
        dtype=str,
        thousands='.', 
        decimal=','
    )
    return df

def update_sheet(spreadsheetId, spread_range, scopes, creds, df):
    # Make a copy to avoid changing the original dataframe
    df_clean = df.copy()
    
    # 1. ALWAYS convert datetime to string FIRST before modifying fillna values
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d')
            
    # 2. Force conversion to base objects and completely clean all NaN/NaT/<NA> types
    df_clean = df_clean.astype(object).fillna('')
    
    # 3. Double-check element-by-element for any sneaky datetime objects missed by type matching
    header = df_clean.columns.tolist()
    data = df_clean.values.tolist()
    
    clean_data = []
    for row in data:
        clean_row = []
        for cell in row:
            if isinstance(cell, (datetime, pd.Timestamp)):
                clean_row.append(cell.strftime('%Y-%m-%d'))
            elif pd.isna(cell):
                clean_row.append('')
            else:
                clean_row.append(cell)
        clean_data.append(clean_row)
        
    values_with_header = [header] + clean_data
    
    try:
        service = build('sheets', 'v4', credentials=creds)
        service.spreadsheets().values().clear(spreadsheetId=spreadsheetId, range=spread_range, body={}).execute()
        
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId, 
            range=spread_range, 
            valueInputOption="USER_ENTERED", 
            body=body
        ).execute()
        print(f"{result.get('updatedCells')} cells updated (sheet replaced).")
        return result
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error

def append_sheet(spreadSheetId, spread_range, scopes, creds, df):
    df_clean = df.fillna('')
    header = df.columns.tolist()
    data = df.values.tolist()
    
    # Combine them: [header] creates a list within a list to match the Sheets API format
    values_with_header = [header] + data
    try:
        values = values_with_header
        service = build('sheets', 'v4', credentials=creds)
        body = {
            'values': values_with_header
        }
        result = service.spreadsheets().values().append(
            spreadsheetId=spreadSheetId, 
            range=spread_range, 
            valueInputOption="USER_ENTERED", 
            body=body).execute()
        print(f"{(result.get('updates').get('updatedCells'))} cells appended.")
        return result

    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
def extract_data_with_gemini(file_path):
    # 1. Setup Gemini API
    # Replace 'YOUR_API_KEY' with your actual Google AI Studio API key
    #genai.configure(api_key="AQ.Ab8RN6I7WqbeUlRXPjXm9C75hCweJNo5M8AOkZgJ2HP2atp1WA")
    genai.configure(api_key="AQ.Ab8RN6K-C_Hpr3I3ra-94nf7F76ILVJFUXD4r61dpwelFN3BcA")
    generation_config = {
        "temperature": 0,
        "top_p": 0.95,
        "response_mime_type": "application/json",
    }
    model = genai.GenerativeModel(
      model_name="gemini-3.5-flash",
      generation_config=generation_config,
    )
    # 2. Load the Excel data
    # Reading as string helps if the Excel is messy or has multiple headers
    df_raw = get_excel_data(file_path, 'update 19 jan 26 (UPtes (2)', "B, G, BY:CV", 2, 1000, 1)
    raw_data_string = df_raw.to_string()

    # 3. Construct the prompt
    prompt = f"""
    Act as a precise data extraction engine. Extract financial metrics from the provided raw Excel dump.

    ### EXTRACTION RULES:
    1. MAPPING: Locate the exact row for each of the 2 metrics listed below.
    2. VALUES: 
       - Extract 'Planning', 'Realisasi' values as numbers.
       - If a value is missing, null, or "-" in the raw data, return 0.0. 
       - DO NOT use nested dictionaries. Use a flat structure.
       - Fill Month Value with January until Desember for each project.
    3. Group the values based on end of Month: "31/01/2026", "28/02/2026", etc
    4. Fill the missing value in PROJECT CODE / WBS or NAMA CAPEX with "Belum terisi"
    4. FORMAT: Return ONLY a valid JSON list of objects. No conversational text.

    ### METRIC LIST:
    1. PROJECT CODE / WBS
    2. NAMA CAPEX
    ### REQUIRED JSON STRUCTURE EXAMPLE:
    [
      {{
        "Project Code": "P6-22270-07-C-12", 
        "Nama Capex": "IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM 5 DAN CM 9 GHOPO TUBAN ", 
        "Month": "31/01/2026", 
        "Planning": 500.000.000,
        "Realisasi": 2.275.000.000 
      }},
      {{
        "Project Code": "P6-22270-07-C-12", 
        "Nama Capex": "IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM 5 DAN CM 9 GHOPO TUBAN ", 
        "Month": "28/02/2026", 
        "Planning": 0,
        "Realisasi": 22.350.000 
      }}
    ]

    Raw Data:
    {raw_data_string}
    """

    # 4. Generate response
    response = model.generate_content(prompt)
    json_text = response.text.replace('```json', '').replace('```', '').strip()

    print(f"DEBUG: Response Length: {len(json_text)}") # See if it's hitting a limit
    try:
        data = json.loads(json_text)
        # 5. Convert back to DataFrame
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None

if __name__ == '__main__':
    main()

DEBUG: Response Length: 39232
         Project Code                                         Nama Capex  \
0    P6-22270-07-C-12  IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM...   
1    P6-22270-07-C-12  IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM...   
2    P6-22270-07-C-12  IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM...   
3    P6-22270-07-C-12  IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM...   
4    P6-22270-07-C-12  IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM...   
..                ...                                                ...   
223          P2-24057              GP-3 REPLACE BEARING ROLLER RM 343RM1   
224          P2-24057              GP-3 REPLACE BEARING ROLLER RM 343RM1   
225          P2-24057              GP-3 REPLACE BEARING ROLLER RM 343RM1   
226          P2-24057              GP-3 REPLACE BEARING ROLLER RM 343RM1   
227          P2-24057              GP-3 REPLACE BEARING ROLLER RM 343RM1   

          Month     Planning     Realisasi  
0    31/01/2

C:\Users\mochamad.ilmawan\AppData\Local\Temp\ipykernel_34524\3087214116.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df_clean.astype(object).fillna('')


1145 cells updated (sheet replaced).
